# 向量嵌入

小明一边读这本书，一边问学习助手问题。可是他很快发现，学习助手对这本书一无所知：书是新写的，大模型训练的时候根本没见过；就算问的是通用的知识，它的说法也常常和书里不一样。

一个直接的想法是：把整本书放进系统消息里。但这行不通：

- 一本书有几十万字，很可能超出大模型一次能接收的上限；
- 就算放得下，每问一个问题都要把整本书发一遍，又慢又贵；
- 内容太多，大模型反而容易抓不住重点。

更好的办法是像人一样：**先翻书找到相关的几段，再根据这几段来回答**。这就是 **RAG**（Retrieval-Augmented Generation，检索增强生成）：

1. **检索**（Retrieval）：根据问题，从资料里找出最相关的几段；
2. **增强**（Augmented）：把这几段放进给大模型的消息里；
3. **生成**（Generation）：大模型参考这些资料，生成回答。

第 13–16 章，我们一步步把 RAG 做出来。这一章先解决最关键的问题：**怎样找到“相关”的内容？**

## 关键词搜索不够用

最容易想到的是关键词搜索：问题里有哪些词，就去找包含这些词的段落。但它有两个明显的问题：

- **说法不同**：小明问“怎样让 AI 使用计算器”，书里写的是“工具调用让模型请求执行外部函数”。意思一样，却几乎没有相同的词。
- **语言不同**：小明用英文提问，书是中文写的，一个词都对不上。

我们需要一种按**意思**而不是按**字面**来比较的方法。

## 向量嵌入

**向量嵌入**（embedding）就是这样一种方法：用一个专门的模型，把一段文字变成一串数字，也就是一个**向量**。这个模型经过大量训练，能做到：

> 意思相近的文字，得到的向量也相近。

可以把每个向量想象成空间里的一个点。“猫坐在垫子上”和“A cat is sitting on the mat”意思一样，它们的点就挨得很近；“今天股市大跌”说的是另一回事，它的点就离得很远。

这样，“找相关内容”就变成了一个数学问题：**找离问题最近的那些点**。

> 这一章用 OpenAI 的嵌入模型。Anthropic 目前没有提供自己的嵌入模型，所以这里不写 Claude 的版本。

导入的部分多了 `numpy`，我们用它来做向量计算。如果还没有安装，运行 `pip install numpy`。

In [ ]:
import asyncio
import inspect
import json
import os
import threading
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [ ]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, url: str):  # the URL of a remote MCP server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(url, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, url: str, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(url) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [ ]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

## EmbeddingProvider

和 `LLMProvider` 一样，我们先定义一个抽象类 `EmbeddingProvider`，它只有一个方法 `embed`：传入一组文字，返回一组向量。

返回值是一个 `numpy` 的二维数组，每一行是一段文字的向量。比如传入 5 段文字，模型生成的向量有 1536 个数字，返回的数组形状就是 `(5, 1536)`。

In [ ]:
class EmbeddingProvider(ABC):

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

`OpenAIEmbedding` 调用 OpenAI 的嵌入接口。默认用的模型是 `text-embedding-3-small`，它又快又便宜，效果也不错。

一次可以传入很多段文字。返回的结果里，每个向量都带着一个编号 `index`，表示它对应第几段文字。我们按编号排好顺序，保证向量和文字一一对应。

In [ ]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

## 余弦相似度

两个向量“近不近”，怎么衡量呢？最常用的是**余弦相似度**（cosine similarity）：看两个向量的**方向**有多接近。

- 方向完全相同，结果是 1；
- 方向互相垂直，结果是 0，表示没什么关系；
- 方向完全相反，结果是 -1。

计算方法分两步：

1. 把每个向量除以它的长度（`np.linalg.norm`），变成长度为 1 的向量。这一步叫**归一化**。
2. 两个长度为 1 的向量，对应位置相乘再加起来（点积），结果就是余弦相似度。

`a @ b.T` 是 `numpy` 的矩阵乘法，一次就算出了 `a` 的每一行和 `b` 的每一行之间的相似度。如果 `a` 有 3 行、`b` 有 7 行，结果就是一个 3×7 的表格。

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

这一章 `Agent` 没有改动。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

## 看看向量长什么样

把 5 句话变成向量，看看形状、长度和前几个数字。

OpenAI 返回的向量已经归一化过了，长度都是 1。每个数字单独看没有什么意义，意思是由 1536 个数字**合在一起**表达的。

In [ ]:
embedder = OpenAIEmbedding()

sentences = [
    "猫坐在垫子上。",
    "A cat is sitting on the mat.",
    "今天股市大跌。",
    "The stock market fell sharply today.",
    "我喜欢用 Python 写程序。",
]
vectors = embedder.embed(sentences)
print("shape:", vectors.shape)                        # (5, 1536)
print("length:", np.linalg.norm(vectors, axis=1))     # OpenAI embeddings are already unit length
print("first 5 numbers:", vectors[0][:5])

计算这 5 句话两两之间的相似度。注意观察：

- 每句话和自己的相似度是 1；
- 中文的“猫坐在垫子上”和英文的“A cat is sitting on the mat”，虽然语言不同，相似度却很高；
- 讲猫的句子和讲股市的句子，相似度就低得多。

> 相似度的具体数值和模型有关。不要死记“多少分算相关”，要看的是**相对高低**。

In [ ]:
similarity = cosine_similarity(vectors, vectors)
for sentence, row in zip(sentences, similarity):
    print("  ".join(f"{score:.2f}" for score in row), "|", sentence)

## 语义搜索

有了嵌入和相似度，就可以做**语义搜索**了：

1. 把所有资料变成向量；
2. 把问题也变成向量；
3. 算出问题和每段资料的相似度；
4. 取相似度最高的 `top_k` 段。

`np.argsort(scores)` 返回从小到大排序后的下标，`[::-1]` 把它倒过来变成从大到小，`[:top_k]` 取前几个。

In [ ]:
def semantic_search(query: str, documents: List[str], embedder: EmbeddingProvider, top_k: int = 3):
    doc_vectors = embedder.embed(documents)
    query_vector = embedder.embed([query])
    scores = cosine_similarity(query_vector, doc_vectors)[0]
    best = np.argsort(scores)[::-1][:top_k]  # indices of the highest scores
    return [(documents[i], float(scores[i])) for i in best]

我们准备几段关于这本书内容的资料，外加一段完全无关的（蒙特利尔的天气）。

In [ ]:
documents = [
    "工具调用让模型可以请求程序执行外部函数，比如查询时间或访问数据库。",
    "记忆模块保存对话历史，当历史太长时会被压缩成一段摘要。",
    "MCP 是一种开放协议，用统一的方式把工具提供给不同的 AI 应用。",
    "流式输出让用户在模型生成的同时，逐字看到回复内容。",
    "向量嵌入把文本映射为高维空间中的点，语义相近的文本距离更近。",
    "Agent 循环不断调用模型和工具，直到模型给出最终答案。",
    "蒙特利尔的冬天很冷，一月平均气温在零下十度左右。",
]

用三个问题来试试。它们都很难用关键词搜索找到答案：第一个和正确答案几乎没有相同的词，第三个干脆是英文。看看语义搜索能不能把正确的资料排在第一位。

In [ ]:
queries = [
    "怎样让 AI 使用计算器这样的外部能力？",   # no shared keywords with the right document
    "对话太长了怎么办？",
    "How can the assistant show its answer while it is still generating?",  # English query, Chinese documents
]
for query in queries:
    print("Q:", query)
    for document, score in semantic_search(query, documents, embedder):
        print(f"   {score:.3f}  {document}")

## 动手试试

- 在 `sentences` 里加几句话，比如“小狗躺在地毯上”和“Python 是一门编程语言”，看看它们和其他句子的相似度。
- 自己想几个关于前 12 章内容的问题，用 `semantic_search` 搜一搜，看看结果是否符合你的预期。哪种问题容易搜错？
- 换一个更大的嵌入模型：`OpenAIEmbedding("text-embedding-3-large")`，看看向量的长度变成多少，相似度有没有变化。

## 小结

- **RAG** 分三步：检索相关资料，放进消息里，让大模型参考资料回答。这样大模型就能回答它没学过的内容。
- 关键词搜索只看字面，**向量嵌入**能按意思比较文字，甚至跨越不同的语言。
- 嵌入模型把文字变成向量，意思相近的文字，向量也相近。
- **余弦相似度**衡量两个向量的方向有多接近：先归一化，再算点积。
- **语义搜索**就是找和问题向量最相似的资料。

不过，现在每次搜索都要把所有资料重新变成向量，而且资料都是一句一句手写的。下一章，我们把真正的书切成小段，提前算好向量，保存成一个**索引**，以后直接查。